# 04 · Modelo de riesgo de abstención para 2027

> **Reto 2 del INE:** "¿Qué zonas podrían presentar mayor abstención en el futuro si no se interviene?"

**A qué responde este paso.** Con la historia limpia de cada municipio (notebook 03) y sus variables (notebook 03b), construimos un modelo que anticipa **qué municipios quedarán más por debajo del país en la intermedia de 2027**, y lo ponemos a prueba **como si estuviéramos antes de una elección que no conocemos**.

**Cómo lo hacemos** (decisiones D12, D14 y D15):

1. **Qué se predice:** la **desviación** de cada municipio frente a la participación nacional. El nivel nacional de 2027 es un **escenario**, no una predicción.
2. **Cómo se valida:** se aprende de una transición presidencial → intermedia y se predice la otra.
   - **La prueba principal es aprender de 2012 → 2015 y predecir 2018 → 2021**, porque respeta el orden del tiempo.
   - También se hace al revés, para medir la incertidumbre.
3. **Contra qué se compara:** la **regla simple justa**, que es la desviación de la presidencial anterior más el cambio típico de la presidencial a la intermedia. Un modelo solo vale si le gana.
4. **Los municipios con parte de su lista sin casilla** se tratan aparte (D12). El año sin casilla no se usa para entrenar ni para medir, y llevan una alerta propia.

## Resumen de lo que encontramos

1. **El modelo le gana a la regla simple, pero no en todo.** Aprendiendo de 2015 y prediciendo 2021:
   - identifica mejor **qué municipios quedan por debajo**: error centrado de 5.3 pp contra 6.0 (−12 %), y de 4.3 contra 6.8 ponderado por personas (−37 %);
   - acierta el 70 % del 20 % de mayor riesgo, contra el 67 % de la regla simple.
2. **Donde no gana:**
   - **El nivel general:** el modelo erró el desplazamiento de 2021. Por eso el nivel de 2027 se presenta como escenario, no como predicción.
   - **El 10 % más extremo:** acierta 57 % contra 61 % de la regla simple.
3. **Al revés** (aprender de 2021 y predecir 2015), la mejora es grande: el error centrado baja 35 %.
4. **El modelo elegido es un ensamble: el promedio de una regresión robusta (Huber) y un random forest, con 34 variables** (D21). Le ganó por poco a Huber solo, que fue el elegido hasta el 20 de octubre de 2026; el orden de riesgo de los dos casi coincide (correlación de 0.997). Lo que más pesa es la historia del municipio y si elige ayuntamiento el mismo día. Gradient boosting transfiere muy mal de una elección a otra y queda descartado.
5. **Riesgo 2027:** el 10 % de mayor riesgo son 248 municipios con 9.2 millones de personas en la lista.
   - 227 de ellos no tendrán elección municipal el mismo día: 197 de Oaxaca (usos y costumbres), 16 de Durango, 8 de Veracruz y 6 que eligen por sistemas normativos indígenas fuera de Oaxaca.
   - También entran ciudades grandes, como Tijuana, Juárez, Mexicali y Reynosa.
6. **Corregimos un sesgo antes de publicar:** los antecedentes de falta de casillas no pueden entrar al modelo, porque se aprendían de los pocos municipios que se recuperaron. Van en la **alerta aparte** (D12).

## Palabras que usamos

| Palabra | Qué significa |
|---|---|
| **Desviación** | Participación del municipio menos la nacional, en puntos porcentuales (pp). |
| **Error medio** | Promedio de cuántos pp se equivoca la predicción en cada municipio. |
| **Error centrado** | El error medio después de quitar el desplazamiento general de la elección. Mide si el modelo acierta **qué municipios** quedan arriba o abajo, sin importar si ese año todo el país votó más o menos. |
| **Error de posición** | Cuántos lugares percentiles se equivoca la predicción, en promedio. Por ejemplo, 10 = el municipio que quedó en el lugar 30 de cada 100 se predijo cerca del 20 o del 40. |
| **Acierto en el 10 % o 20 % de mayor riesgo** | De los municipios que de verdad quedaron entre los que menos votaron, qué parte identificó el modelo. Es lo que más le importa al INE para decidir dónde intervenir. |
| **Ridge y Huber** | Regresiones lineales con protección. Ridge reparte el peso entre variables que dicen lo mismo; Huber además resiste los casos extremos. |
| **Random forest** | Cientos de árboles de decisión, cada uno con una muestra distinta de municipios y variables, cuyas predicciones se promedian. |
| **Ensamble** | Promedio de las predicciones de dos modelos distintos, para que los errores de uno compensen los del otro. |
| **Bootstrap** | Repetir el entrenamiento muchas veces con muestras al azar de municipios, para ver cuánto cambia la predicción: esa es su incertidumbre. |

In [1]:
from pathlib import Path
import warnings

import numpy as np
import polars as pl
from scipy.stats import rankdata
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV, HuberRegressor, LinearRegression
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor

warnings.filterwarnings('ignore')
pl.Config.set_tbl_rows(60)
pl.Config.set_tbl_cols(16)
pl.Config.set_fmt_str_lengths(80)
pl.Config.set_tbl_width_chars(200)

raiz = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').is_file() and (p / 'data').is_dir())
variables = pl.read_parquet(raiz / 'data/processed/variables_transiciones.parquet')
indicadores = pl.read_parquet(raiz / 'data/processed/indicadores_mapa.parquet')
panel = pl.read_parquet(raiz / 'data/processed/panel_municipal.parquet')

# Los antecedentes de falta de casilla NO entran al modelo continuo: van en la alerta aparte (D12). Ver la sección 1.
HISTORIA = ['desviacion_presidencial', 'desviacion_intermedia_previa', 'vecinos_desviacion_presidencial']
LOCALES = ['gobernador_mismo_dia_intermedia', 'diputacion_local_mismo_dia_intermedia', 'ayuntamiento_mismo_dia_intermedia',
           'gobernador_mismo_dia_presidencial', 'diputacion_local_mismo_dia_presidencial', 'ayuntamiento_mismo_dia_presidencial', 'es_sni']
POLITICAS = ['margen_presidencial', 'partidos_efectivos_presidencial']
LISTA = ['pct_rural', 'pct_hombres', 'pct_20_29', 'pct_65_mas', 'crecimiento_lista', 'cobertura_lista_20mas']
SOCIALES = ['escolaridad_promedio', 'pct_habla_lengua_indigena', 'pct_viviendas_internet', 'pct_residia_otro_pais_hace5', 'pct_residia_otra_entidad_hace5',
            'poblacion_log', 'pct_pobreza', 'pct_pobreza_extrema', 'pct_carencia_salud', 'pct_rezago_educativo', 'marginacion', 'pct_localidades_menos_5000',
            'pct_viviendas_emigrantes', 'pct_viviendas_remesas', 'pct_viviendas_retorno', 'homicidios_100mil_log']
# Estables en las dos transiciones pasadas (notebook 03b, sección 5)
ESTABLES = ['pct_hombres', 'cobertura_lista_20mas', 'pct_rezago_educativo', 'pct_residia_otra_entidad_hace5', 'pct_65_mas', 'escolaridad_promedio',
            'crecimiento_lista', 'pct_rural', 'pct_carencia_salud', 'pct_habla_lengua_indigena', 'pct_localidades_menos_5000', 'marginacion',
            'pct_viviendas_internet', 'poblacion_log', 'pct_20_29']
TODAS = HISTORIA + LOCALES + POLITICAS + LISTA + SOCIALES

def con_resultado(T):
    # Municipios con resultado conocido, sin el año en que les faltó casilla (D12) y con desviación presidencial
    return variables.filter((pl.col('eleccion_objetivo') == T) & pl.col('desviacion_objetivo').is_not_null()
                            & ~pl.col('sin_casilla_objetivo').fill_null(False) & pl.col('desviacion_presidencial').is_not_null())

entrenar_2015, entrenar_2021 = con_resultado(2015), con_resultado(2021)
print(f'Municipios con resultado conocido: 2015 = {entrenar_2015.height:,} · 2021 = {entrenar_2021.height:,}')

Municipios con resultado conocido: 2015 = 2,418 · 2021 = 2,437


## 1 · Los candidatos

| Modelo | Variables | Idea |
|---|---|---|
| **Regla simple justa** | Desviación de la presidencial anterior | "El municipio repite su posición, más el cambio típico de la presidencial a la intermedia" |
| **Lineal: historia** | Historia del municipio | ¿Cuánto aporta ajustar el peso de la historia? |
| **Ridge: historia + elecciones locales** | + qué se elige el mismo día | La variable más estable (notebook 03b) |
| **Ridge: variables estables** | + solo las variables sociales y de la lista que se sostuvieron en las dos transiciones | Más prudente |
| **Ridge: todas** | Todas las variables | Lo deja todo al modelo, con regularización |
| **Huber: todas** | Todas, resistente a casos extremos | Igual, pero no se deja arrastrar por municipios atípicos |
| **Gradient boosting: todas** | Todas, con árboles | Modelo no lineal, para comparar |
| **Random forest: todas** | Todas, con cientos de árboles promediados | Otro modelo no lineal, más estable que el boosting |
| **Ensamble Huber + random forest** | Promedio de las dos predicciones | Huber capta lo general; el bosque, combinaciones que un modelo lineal no ve (D21) |

**Por qué los antecedentes de falta de casilla no entran al modelo.** En una primera versión los incluimos (si al municipio le faltó casilla en la presidencial y cuántas veces le ha faltado). El modelo los aprendió de **solo 4 municipios**: los que sí tuvieron casillas en la intermedia siguiente, porque D12 deja fuera del entrenamiento los años sin casilla. Esos 4 se recuperaron mucho, así que el modelo concluyó que "le faltó casilla" anticipa un **rebote de +29 puntos**. Así, Tila, Nahuatzen o Charapan salían como los municipios de **menor** riesgo del país. Es un sesgo de selección. Por eso esos antecedentes van solo en la **alerta "sin casilla"** (D12), separada del modelo.

In [2]:
def pipeline(tipo):
    pasos = [SimpleImputer(strategy='median'), StandardScaler()]
    if tipo == 'lineal': return make_pipeline(*pasos, LinearRegression())
    if tipo == 'ridge': return make_pipeline(*pasos, RidgeCV(alphas=np.logspace(-2, 4, 30)))
    if tipo == 'huber': return make_pipeline(*pasos, HuberRegressor(alpha=1.0, epsilon=1.35, max_iter=3000))
    if tipo == 'arboles': return HistGradientBoostingRegressor(max_iter=300, learning_rate=0.04, max_leaf_nodes=15, min_samples_leaf=40, l2_regularization=1.0, random_state=0)
    # Ajustes del bosque: los que eligió la validación cruzada interna en las dos direcciones (notebook 04b, sección 5)
    if tipo == 'bosque': return make_pipeline(SimpleImputer(strategy='median'), RandomForestRegressor(n_estimators=300, min_samples_leaf=5, max_features=0.6, random_state=0, n_jobs=-1))
    if tipo == 'ensamble': return Ensamble()

class Ensamble:
    # Promedio simple de las predicciones de Huber y de random forest (D21)
    def fit(self, X, y):
        self.partes = [pipeline('huber').fit(X, y), pipeline('bosque').fit(X, y)]
        return self
    def predict(self, X):
        return np.mean([p.predict(X) for p in self.partes], axis=0)

CANDIDATOS = {
    'Lineal: historia': ('lineal', HISTORIA),
    'Ridge: historia + elecciones locales': ('ridge', HISTORIA + LOCALES),
    'Ridge: variables estables': ('ridge', HISTORIA + LOCALES + ESTABLES),
    'Ridge: todas': ('ridge', TODAS),
    'Huber: todas': ('huber', TODAS),
    'Gradient boosting: todas': ('arboles', TODAS),
    'Random forest: todas': ('bosque', TODAS),
    'Ensamble Huber + random forest: todas': ('ensamble', TODAS),
}

def percentil(x):
    return (rankdata(x) - 0.5) / len(x) * 100

def metricas(y, p, lista):
    k10, k20 = len(y) // 10, len(y) // 5
    bajos = lambda v, k: set(np.argsort(v)[:k])
    centrado = np.abs((y - np.median(y)) - (p - np.median(p)))
    return {
        'R²': 1 - ((y - p) ** 2).sum() / ((y - y.mean()) ** 2).sum(),
        'Error medio (pp)': np.abs(y - p).mean(),
        'Error centrado (pp)': centrado.mean(),
        'Error centrado ponderado por lista (pp)': np.average(centrado, weights=lista),
        'Error de posición (percentiles)': np.abs(percentil(y) - percentil(p)).mean(),
        'Acierto 10 % de mayor riesgo': len(bajos(y, k10) & bajos(p, k10)) / k10,
        'Acierto 20 % de mayor riesgo': len(bajos(y, k20) & bajos(p, k20)) / k20,
    }

def validar(A, B):
    y, lista = B['desviacion_objetivo'].to_numpy(), B['lista_presidencial'].to_numpy().astype(float)
    filas = []
    cambio = (A['desviacion_objetivo'] - A['desviacion_presidencial']).median()
    filas.append({'Modelo': 'Regla simple justa', **metricas(y, B['desviacion_presidencial'].to_numpy() + cambio, lista)})
    predicciones = {}
    for nombre, (tipo, cols) in CANDIDATOS.items():
        m = pipeline(tipo).fit(A.select(cols).to_numpy().astype(float), A['desviacion_objetivo'].to_numpy())
        p = m.predict(B.select(cols).to_numpy().astype(float))
        predicciones[nombre] = p
        filas.append({'Modelo': nombre, **metricas(y, p, lista)})
    return pl.DataFrame(filas).with_columns(pl.exclude('Modelo').round(3)), predicciones

## 2 · Prueba principal: aprender de 2012 → 2015 y predecir 2018 → 2021

Es la situación más parecida a la de 2027: el modelo solo conoce el pasado y predice una intermedia que no ha visto.

In [3]:
resultado_adelante, pred_2021 = validar(entrenar_2015, entrenar_2021)
resultado_adelante

Modelo,R²,Error medio (pp),Error centrado (pp),Error centrado ponderado por lista (pp),Error de posición (percentiles),Acierto 10 % de mayor riesgo,Acierto 20 % de mayor riesgo
str,f64,f64,f64,f64,f64,f64,f64
"""Regla simple justa""",0.682,6.133,6.027,6.797,11.759,0.613,0.667
"""Lineal: historia""",0.564,7.58,7.931,7.203,15.984,0.58,0.62
"""Ridge: historia + elecciones locales""",0.541,8.027,6.441,7.346,13.297,0.58,0.678
"""Ridge: variables estables""",0.725,5.722,5.735,4.812,11.059,0.593,0.69
"""Ridge: todas""",0.678,6.399,5.793,4.785,11.23,0.572,0.7
"""Huber: todas""",0.651,6.73,5.487,4.682,10.561,0.584,0.715
"""Gradient boosting: todas""",0.259,9.661,7.323,5.564,15.286,0.214,0.421
"""Random forest: todas""",0.499,8.121,5.827,4.913,11.244,0.436,0.628
"""Ensamble Huber + random forest: todas""",0.611,7.142,5.317,4.296,10.178,0.568,0.698


**Cómo leer la tabla:** en todas las columnas de error, **menos es mejor**; en R² y en aciertos, **más es mejor**. La fila que importa comparar es cada modelo contra la **regla simple justa**.

## 3 · Prueba al revés: aprender de 2018 → 2021 y predecir 2012 → 2015

No respeta el orden del tiempo, pero sirve para ver si las conclusiones se sostienen en la otra dirección. 2015 es la intermedia más difícil: tiene el boicot magisterial y poca concurrencia local.

In [4]:
resultado_atras, pred_2015 = validar(entrenar_2021, entrenar_2015)
resultado_atras

Modelo,R²,Error medio (pp),Error centrado (pp),Error centrado ponderado por lista (pp),Error de posición (percentiles),Acierto 10 % de mayor riesgo,Acierto 20 % de mayor riesgo
str,f64,f64,f64,f64,f64,f64,f64
"""Regla simple justa""",0.451,9.356,9.427,7.621,18.53,0.328,0.453
"""Lineal: historia""",0.46,8.973,9.105,7.338,18.037,0.344,0.462
"""Ridge: historia + elecciones locales""",0.622,7.611,7.175,6.723,14.173,0.427,0.547
"""Ridge: variables estables""",0.706,6.358,6.363,4.899,12.679,0.436,0.582
"""Ridge: todas""",0.712,6.346,6.449,4.879,12.641,0.44,0.584
"""Huber: todas""",0.689,6.538,6.567,4.903,12.71,0.436,0.582
"""Gradient boosting: todas""",0.688,6.836,6.738,5.415,13.046,0.465,0.557
"""Random forest: todas""",0.688,6.98,6.153,5.635,11.919,0.461,0.598
"""Ensamble Huber + random forest: todas""",0.744,6.152,6.158,4.76,11.891,0.465,0.594


## 4 · Elegimos el modelo

Comparamos los dos sentidos lado a lado, con las tres medidas que más importan para el reto: el error centrado (qué municipios quedan abajo), el error de posición y el acierto en el 20 % de mayor riesgo.

In [5]:
claves = ['Error centrado (pp)', 'Error de posición (percentiles)', 'Acierto 20 % de mayor riesgo', 'Acierto 10 % de mayor riesgo']
comparacion = (resultado_adelante.select('Modelo', *[pl.col(c).alias(f'{c} · 2015→2021') for c in claves])
               .join(resultado_atras.select('Modelo', *[pl.col(c).alias(f'{c} · 2021→2015') for c in claves]), on='Modelo'))
base = comparacion.filter(pl.col('Modelo') == 'Regla simple justa')
comparacion.with_columns(
    mejora_error_adelante=((1 - pl.col('Error centrado (pp) · 2015→2021') / base['Error centrado (pp) · 2015→2021'][0]) * 100).round(1),
    mejora_error_atras=((1 - pl.col('Error centrado (pp) · 2021→2015') / base['Error centrado (pp) · 2021→2015'][0]) * 100).round(1),
).rename({'mejora_error_adelante': 'Mejora del error centrado vs. regla simple, 2015→2021 (%)', 'mejora_error_atras': 'Ídem, 2021→2015 (%)'})

Modelo,Error centrado (pp) · 2015→2021,Error de posición (percentiles) · 2015→2021,Acierto 20 % de mayor riesgo · 2015→2021,Acierto 10 % de mayor riesgo · 2015→2021,Error centrado (pp) · 2021→2015,Error de posición (percentiles) · 2021→2015,Acierto 20 % de mayor riesgo · 2021→2015,Acierto 10 % de mayor riesgo · 2021→2015,"Mejora del error centrado vs. regla simple, 2015→2021 (%)","Ídem, 2021→2015 (%)"
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""Regla simple justa""",6.027,11.759,0.667,0.613,9.427,18.53,0.453,0.328,0.0,0.0
"""Lineal: historia""",7.931,15.984,0.62,0.58,9.105,18.037,0.462,0.344,-31.6,3.4
"""Ridge: historia + elecciones locales""",6.441,13.297,0.678,0.58,7.175,14.173,0.547,0.427,-6.9,23.9
"""Ridge: variables estables""",5.735,11.059,0.69,0.593,6.363,12.679,0.582,0.436,4.8,32.5
"""Ridge: todas""",5.793,11.23,0.7,0.572,6.449,12.641,0.584,0.44,3.9,31.6
"""Huber: todas""",5.487,10.561,0.715,0.584,6.567,12.71,0.582,0.436,9.0,30.3
"""Gradient boosting: todas""",7.323,15.286,0.421,0.214,6.738,13.046,0.557,0.465,-21.5,28.5
"""Random forest: todas""",5.827,11.244,0.628,0.436,6.153,11.919,0.598,0.461,3.3,34.7
"""Ensamble Huber + random forest: todas""",5.317,10.178,0.698,0.568,6.158,11.891,0.594,0.465,11.8,34.7


In [6]:
# Regla de elección: el modelo con menor error centrado promedio en los dos sentidos, que además le gane a la regla simple en los dos
promedio = comparacion.with_columns(error_promedio=(pl.col('Error centrado (pp) · 2015→2021') + pl.col('Error centrado (pp) · 2021→2015')) / 2)
le_gana = promedio.filter(
    (pl.col('Error centrado (pp) · 2015→2021') < base['Error centrado (pp) · 2015→2021'][0])
    & (pl.col('Error centrado (pp) · 2021→2015') < base['Error centrado (pp) · 2021→2015'][0]))
MODELO = le_gana.sort('error_promedio')['Modelo'][0]
TIPO, COLUMNAS = CANDIDATOS[MODELO]
print(f'Modelos que le ganan a la regla simple en los dos sentidos: {le_gana["Modelo"].to_list()}')
print(f'Modelo elegido: {MODELO} ({len(COLUMNAS)} variables)')

Modelos que le ganan a la regla simple en los dos sentidos: ['Ridge: variables estables', 'Ridge: todas', 'Huber: todas', 'Random forest: todas', 'Ensamble Huber + random forest: todas']
Modelo elegido: Ensamble Huber + random forest: todas (34 variables)


## 5 · Qué pesa en el modelo

Entrenamos el modelo elegido con **las dos transiciones juntas** (2015 y 2021). Así aprende de un régimen con poca concurrencia local y de otro con mucha, y 2027 se parece al segundo. Mostramos el peso de cada variable en la parte lineal (Huber), ya estandarizada: cuántos pp cambia la desviación prevista cuando la variable sube una desviación estándar, con todo lo demás igual. La parte de random forest no tiene pesos; su importancia se mide por permutación en el notebook 04b (sección 4).

In [7]:
juntos = pl.concat([entrenar_2015, entrenar_2021], how='diagonal_relaxed')
modelo_final = pipeline(TIPO).fit(juntos.select(COLUMNAS).to_numpy().astype(float), juntos['desviacion_objetivo'].to_numpy())
parte_lineal = modelo_final.partes[0] if isinstance(modelo_final, Ensamble) else modelo_final
coeficientes = parte_lineal[-1].coef_ if hasattr(parte_lineal[-1], 'coef_') else None
if coeficientes is not None:
    print('Pesos de la parte lineal (Huber) del modelo:' if isinstance(modelo_final, Ensamble) else 'Pesos del modelo:')
    pesos = (pl.DataFrame({'Variable': COLUMNAS, 'Peso (pp por desviación estándar)': coeficientes})
             .with_columns(pl.col('Peso (pp por desviación estándar)').round(2))
             .sort(pl.col('Peso (pp por desviación estándar)').abs(), descending=True))
    print(pesos.head(20))

Pesos de la parte lineal (Huber) del modelo:
shape: (20, 2)
┌───────────────────────────────────────┬───────────────────────────────────┐
│ Variable                              ┆ Peso (pp por desviación estándar) │
│ ---                                   ┆ ---                               │
│ str                                   ┆ f64                               │
╞═══════════════════════════════════════╪═══════════════════════════════════╡
│ ayuntamiento_mismo_dia_intermedia     ┆ 9.64                              │
│ desviacion_presidencial               ┆ 8.57                              │
│ ayuntamiento_mismo_dia_presidencial   ┆ -3.56                             │
│ poblacion_log                         ┆ -2.91                             │
│ marginacion                           ┆ 2.22                              │
│ pct_pobreza_extrema                   ┆ 1.54                              │
│ es_sni                                ┆ -1.12                             │
│ pc

**Cuidado al leer los pesos:** muchas variables dicen lo mismo (notebook 03b, sección 6), así que el modelo reparte el peso entre ellas. Un peso individual no es "el efecto" de esa variable; lo confiable es el **conjunto**.

## 6 · El riesgo de cada municipio en 2027

1. **Desviación prevista:** el modelo final aplicado a las variables de la transición 2024 → 2027.
2. **Incertidumbre:** 300 repeticiones del entrenamiento con muestras al azar de municipios (bootstrap). Reportamos el intervalo del 90 % de la desviación prevista. Este intervalo **no** incluye el error propio de una elección nueva, que medimos en la validación (sección 2).
3. **Participación prevista en tres escenarios nacionales**, que son las intermedias anteriores: **bajo** 44.1 % (2009), **medio** 47.1 % (2015) y **alto** 51.8 % (2021). La participación de cada municipio = escenario + su desviación prevista, ajustada para que el promedio nacional ponderado por la lista coincida con el escenario.
4. **Posición de riesgo:** percentil 0 = el municipio que se prevé más por debajo del país.

In [8]:
futuro = variables.filter(pl.col('eleccion_objetivo') == 2027)
X_futuro = futuro.select(COLUMNAS).to_numpy().astype(float)
desviacion_prevista = modelo_final.predict(X_futuro)

rng = np.random.default_rng(2027)
X_entrena, y_entrena = juntos.select(COLUMNAS).to_numpy().astype(float), juntos['desviacion_objetivo'].to_numpy()
repeticiones = []
for _ in range(300):
    i = rng.integers(0, len(y_entrena), len(y_entrena))
    repeticiones.append(pipeline(TIPO).fit(X_entrena[i], y_entrena[i]).predict(X_futuro))
repeticiones = np.array(repeticiones)

lista_2024 = panel.filter(pl.col('AELEC') == 2024).select('cvegeo', lista_2024='LN')
riesgo = futuro.select('cvegeo', 'estado', 'municipio').with_columns(
    desviacion_prevista=pl.Series(desviacion_prevista),
    desviacion_p05=pl.Series(np.percentile(repeticiones, 5, axis=0)),
    desviacion_p95=pl.Series(np.percentile(repeticiones, 95, axis=0)),
).join(lista_2024, on='cvegeo', how='left')

ESCENARIOS = {'bajo': 44.06, 'medio': 47.07, 'alto': 51.84}
pesos_lista = riesgo['lista_2024'].fill_null(0).to_numpy().astype(float)
for nombre, nivel in ESCENARIOS.items():
    bruta = nivel + riesgo['desviacion_prevista'].to_numpy()
    ajuste = nivel - np.average(bruta, weights=pesos_lista)
    riesgo = riesgo.with_columns(pl.Series(f'participacion_prevista_{nombre}', np.clip(bruta + ajuste, 0, 100)))
riesgo = (riesgo.with_columns(percentil_riesgo=pl.Series(percentil(riesgo['desviacion_prevista'].to_numpy())))
          .join(indicadores.select('cvegeo', 'alerta_sin_casilla', 'ultima_eleccion_sin_casilla', 'elecciones_por_debajo_del_pais',
                                   'zona_participacion_2024', 'credencializacion_18_19_2021', 'brecha_jovenes_2024',
                                   'ayuntamiento_mismo_dia_2027', 'gobernador_mismo_dia_2027', 'es_sni'), on='cvegeo', how='left')
          .sort('desviacion_prevista'))
riesgo.write_parquet(raiz / 'data/processed/riesgo_2027.parquet')
print(f'Riesgo 2027 guardado para {riesgo.height:,} municipios · sin predicción: {riesgo["desviacion_prevista"].null_count()}')

# Predicciones de la validación del modelo elegido: el paso 5 las usa para el intervalo de 2027
pl.concat([
    B.select('cvegeo', 'lista_presidencial', real='desviacion_objetivo').with_columns(prevista=pl.Series(pred[MODELO]), validacion=pl.lit(nombre))
    for B, pred, nombre in [(entrenar_2021, pred_2021, 'aprende de 2015, predice 2021'), (entrenar_2015, pred_2015, 'aprende de 2021, predice 2015')]
]).write_parquet(raiz / 'data/processed/validacion_modelo.parquet')

Riesgo 2027 guardado para 2,477 municipios · sin predicción: 0


In [9]:
(riesgo.head(25).select(
    'estado', 'municipio',
    pl.col('desviacion_prevista').round(1).alias('Desviación prevista (pp)'),
    pl.format('[{}, {}]', pl.col('desviacion_p05').round(1), pl.col('desviacion_p95').round(1)).alias('Intervalo 90 %'),
    pl.col('participacion_prevista_medio').round(1).alias('Participación prevista, escenario medio (%)'),
    pl.col('lista_2024').alias('Lista 2024'),
    pl.col('alerta_sin_casilla').alias('Alerta "sin casilla"'),
    pl.col('zona_participacion_2024').alias('Zona 2024'),
))

estado,municipio,Desviación prevista (pp),Intervalo 90 %,"Participación prevista, escenario medio (%)",Lista 2024,"Alerta ""sin casilla""",Zona 2024
str,str,f64,str,f64,i64,str,str
"""Oaxaca""","""La Reforma""",-39.0,"""[-40.2, -36.5]""",8.4,2523,"""antecedente""",null
"""Oaxaca""","""Capulálpam de Méndez""",-36.8,"""[-38.5, -34.7]""",10.6,1215,"""antecedente""",null
"""Michoacán de Ocampo""","""Cherán""",-36.3,"""[-37.3, -34.2]""",11.1,15001,"""recurrente""",null
"""Oaxaca""","""San Mateo del Mar""",-33.0,"""[-34.1, -30.7]""",14.4,10676,"""recurrente""",null
"""Oaxaca""","""San Lucas Quiaviní""",-30.9,"""[-32.2, -28.9]""",16.6,1505,"""sin antecedente""","""fría (bajo-bajo)"""
"""Oaxaca""","""San Agustín Yatareni""",-30.6,"""[-31.7, -28.9]""",16.8,3718,"""sin antecedente""","""no significativa"""
"""Oaxaca""","""Santiago Amoltepec""",-30.1,"""[-31.0, -27.9]""",17.4,7910,"""sin antecedente""","""fría (bajo-bajo)"""
"""Oaxaca""","""San Lucas Camotlán""",-29.6,"""[-31.4, -27.3]""",17.8,2086,"""sin antecedente""","""fría (bajo-bajo)"""
"""Oaxaca""","""Magdalena Teitipac""",-29.4,"""[-30.5, -27.1]""",18.0,3313,"""sin antecedente""","""fría (bajo-bajo)"""


In [10]:
# ¿Dónde se concentra el riesgo? Municipios en el 10 % de mayor riesgo, por estado, y personas en ellos
alto_riesgo = riesgo.filter(pl.col('percentil_riesgo') <= 10)
print(f'Municipios en el 10 % de mayor riesgo: {alto_riesgo.height} · personas en su lista (2024): {alto_riesgo["lista_2024"].sum():,}')
(alto_riesgo.group_by('estado').agg(pl.len().alias('Municipios'), pl.col('lista_2024').sum().alias('Personas en la lista'),
                                    (~pl.col('ayuntamiento_mismo_dia_2027')).sum().alias('Sin elección municipal en 2027'))
 .sort('Municipios', descending=True).head(12))

Municipios en el 10 % de mayor riesgo: 248 · personas en su lista (2024): 9,195,856


estado,Municipios,Personas en la lista,Sin elección municipal en 2027
str,u32,i64,u32
"""Oaxaca""",197,659962,197
"""Durango""",16,1083342,16
"""Veracruz de Ignacio de la Llave""",8,1015539,8
"""Nuevo León""",7,860913,0
"""Michoacán de Ocampo""",5,356541,1
"""Baja California""",4,2719824,0
"""Chiapas""",2,67817,1
"""Sonora""",2,385211,0
"""Guerrero""",2,47418,2


## 7 · Resumen y advertencias

### Qué tan bueno es

| Medida (aprende de 2015, predice 2021) | Regla simple justa | Huber solo | Modelo (ensamble) |
|---|---:|---:|---:|
| Error centrado (pp) | 6.03 | 5.49 | **5.32** |
| Error centrado ponderado por personas (pp) | 6.80 | 4.68 | **4.30** |
| Error de posición (percentiles) | 11.8 | 10.6 | **10.2** |
| Acierto en el 20 % de mayor riesgo | 67 % | **72 %** | 70 % |
| Acierto en el 10 % de mayor riesgo | **61 %** | 58 % | 57 % |
| Error sin centrar (pp) | **6.13** | 6.73 | 7.14 |

**Lectura honesta:**

- **El modelo es mejor para decidir dónde poner atención**, sobre todo si se cuentan personas: el error ponderado baja 37 % frente a la regla simple.
- **Frente a Huber solo, el ensamble gana en error y posición, y pierde un poco de acierto en el 20 % de mayor riesgo** (unos 8 municipios de 487). Al predecir 2015 gana en todo.
- **No es mejor para adivinar el nivel general**, porque las relaciones aprendidas en 2015 (poca elección local el mismo día) exageran en 2021. Por eso 2027 se presenta con **tres escenarios nacionales**.
- **En el 10 % más extremo, la regla simple acierta un poco más.** Para el semáforo conviene trabajar con el 20 % de mayor riesgo, donde el modelo sí gana.

### Qué muestra para 2027

- **El riesgo se concentra donde no habrá elección municipal el mismo día:** Oaxaca (usos y costumbres), Durango y Veracruz. Es consistente con lo que vimos en Oaxaca (notebook 03b, sección 3) y con la estabilidad de esa variable en las dos transiciones.
- **También hay riesgo en ciudades grandes de la frontera norte y en zonas metropolitanas**, como Tijuana, Juárez (Chihuahua), Mexicali, Reynosa y Juárez (Nuevo León). Ahí está mucha gente.
- **Los municipios con alerta "sin casilla"** se reportan aparte. Algunos quedan con riesgo alto en el modelo (Cherán, San Mateo del Mar, Chicomuselo, Pantelhó) y otros con riesgo bajo, porque en 2024 sí tuvieron casillas y votaron bien (Bolaños, Mezquitic). La alerta es la que avisa del riesgo de que no haya dónde votar.

### Advertencias

1. **El intervalo del bootstrap es estrecho** (1.7 pp en el municipio típico). Mide cuánto cambia la predicción con otros municipios de entrenamiento, **no** el error propio de una elección nueva. Ese error, medido en la validación, es de unos 5.3 pp por municipio; el notebook 05 lo usa para el intervalo del mapa.
2. **La participación prevista más baja es la de La Reforma y Capulálpam de Méndez**, en Oaxaca (8 % y 11 % en el escenario medio). En 2024 no tuvieron ninguna casilla. Es una alerta, no una certeza. Con Huber solo salían en 0 %; random forest modera los extremos.
3. **El efecto de la elección municipal se aprendió de dos transiciones.** Si en 2027 cambia algo en ese patrón (por ejemplo, una campaña fuerte del INE en esos municipios), el riesgo cambiaría.
4. **Este orden es por municipio.** El INE también necesita la vista por **personas**: un municipio grande con riesgo medio puede tener más abstencionistas que diez pequeños con riesgo alto. Eso se arma en el paso 5 (semáforo y escenarios).

## Siguiente paso

**Paso 5 · Semáforo y escenarios:**

- definir los cortes del semáforo, con el 20 % de mayor riesgo como rojo, donde el modelo se validó mejor;
- sumar la vista por personas;
- construir el escenario "con campaña dirigida" a jóvenes, con primerizos y credencialización.